In [ ]:
## StyleGAN for generating syntetic pairs of overlayed single cell hiPSC-CMs and their accomponying heatmaps


In [2]:
from google.colab import drive
import os
drive.mount("/content/drive")

bf_dir = "/content/drive/MyDrive/BF_HM_FL/AlignedOutputs/overlay/"       # BF w sarcomere overlays
hm_dir = "/content/drive/MyDrive/BF_HM_FL/NoContourHM_converted/"        # HM accompnaying heatmap of BFs

out_root = "/content/drive/MyDrive/stylegan2ada_bf_hm_pairs"
pairs_dir = f"{out_root}/pairs_images"     # paired combo images
os.makedirs(pairs_dir, exist_ok=True)

print("BF:", bf_dir)
print("HM:", hm_dir)
print("OUT:", out_root)


Mounted at /content/drive
BF: /content/drive/MyDrive/BF_HM_FL/AlignedOutputs/overlay/
HM: /content/drive/MyDrive/BF_HM_FL/NoContourHM_converted/
OUT: /content/drive/MyDrive/stylegan2ada_bf_hm_pairs


In [ ]:
dataset_zip_path = "/content/drive/MyDrive/bf_hm_combined_dataset.zip" # zip fn at bootom
training_output_dir = "/content/drive/MyDrive/stylegan2ada_training_results" # training results for checkpoints and the like

In [7]:
# exploring previous runs, choose a snapshot to resume training if previously done based on
#!ls {training_output_dir}
!ls /content/drive/MyDrive/stylegan2ada_training_results/00023-bf_hm_combined_dataset-auto1-gamma8-kimg25000-batch4-ada-resumecustom
resume_path='/content/drive/MyDrive/stylegan2ada_training_results/00023-bf_hm_combined_dataset-auto1-gamma8-kimg25000-batch4-ada-resumecustom/network-snapshot-000080.pkl' # t
print(resume_path)

events.out.tfevents.1772891888.f172f4afb68d.1670.0  network-snapshot-000000.pkl
fakes000000.png					    network-snapshot-000020.pkl
fakes000020.png					    network-snapshot-000040.pkl
fakes000040.png					    network-snapshot-000060.pkl
fakes000060.png					    network-snapshot-000080.pkl
fakes000080.png					    reals.png
fakes_init.png					    stats.jsonl
log.txt						    training_options.json
metric-fid50k_full.jsonl
/content/drive/MyDrive/stylegan2ada_training_results/00023-bf_hm_combined_dataset-auto1-gamma8-kimg25000-batch4-ada-resumecustom/network-snapshot-000080.pkl


In [8]:
import sys
import os
import subprocess

def run_command(command): # debugstep
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    stdout, stderr = process.communicate()
    if process.returncode != 0:
        print(f"Error executing command: {command}")
        print(stderr.decode())
    else:
        print(stdout.decode())
# install style GAN
print("Installing Stylegan2-ada pytorch implementation ")
if not os.path.exists("/content/stylegan2-ada-pytorch"):
    print("Cloning StyleGAN2-ADA repository:")
    run_command("git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git /content/stylegan2-ada-pytorch") # check if it updates
else:
    print("Repository already exists.")

print("\n Installing Dependencies")
print("Installing ninja")
run_command("apt-get install ninja-build -y")
run_command(f"{sys.executable} -m pip install ninja")

print("\n Patching Files for PyTorch 2 Compatibility ") #  have to do some patching for running in colab
# speciffically: grid_sample_gradfix.py conv2d_gradfix.py upfirdn2d.py bias_act.py

def patch_file(path, replacements):
    if not os.path.exists(path):
        print(f"Warning: {path} not found, skipping.")
        return

    with open(path, 'r') as f:
        content = f.read()

    modified = False
    for old, new in replacements:
        if old in content:
            content = content.replace(old, new)
            modified = True

    if modified:
        with open(path, 'w') as f:
            f.write(content)
        print(f"Patched {os.path.basename(path)}")
    else:
        print(f"No changes needed for {os.path.basename(path)}")

# Patch grid_sample_gradfix.py
grid_path = "/content/stylegan2-ada-pytorch/torch_utils/ops/grid_sample_gradfix.py"
grid_replacements = [
    ("enabled = False", "enabled = True"),
    ("['1.7.', '1.8.', '1.9']", "['1.7.', '1.8.', '1.9', '2.']"),
    ("op = torch._C._jit_get_operation('aten::grid_sampler_2d_backward')", "op = torch._C._jit_get_operation('aten::grid_sampler_2d_backward')[0]"),
    ("grad_input, grad_grid = op(grad_output, input, grid, 0, 0, False)", "grad_input, grad_grid = op(grad_output, input, grid, 0, 0, False, (True, True))"),
    ("warnings.warn(f'grid_sample_gradfix not supported on PyTorch {torch.__version__}. Falling back to torch.nn.functional.grid_sample().')", "pass # Warning suppressed")
]
patch_file(grid_path, grid_replacements)

# Patch conv2d_gradfix.py
conv2d_path = "/content/stylegan2-ada-pytorch/torch_utils/ops/conv2d_gradfix.py"
conv2d_replacements = [
    ("warnings.warn(f'conv2d_gradfix not supported on PyTorch {torch.__version__}. Falling back to torch.nn.functional.conv2d().')", "pass # Warning suppressed")
]
patch_file(conv2d_path, conv2d_replacements)

# Patch upfirdn2d.py
upfirdn_path = "/content/stylegan2-ada-pytorch/torch_utils/ops/upfirdn2d.py"
upfirdn_replacements = [
    ("if not _inited:", "if not _inited:\n        _inited = True"),
    ("warnings.warn('Failed to build CUDA kernels for upfirdn2d. Falling back to slow reference implementation. Details:\n\n' + traceback.format_exc())", "pass # Warning suppressed")
]
patch_file(upfirdn_path, upfirdn_replacements)

# Patch bias_act.py
bias_path = "/content/stylegan2-ada-pytorch/torch_utils/ops/bias_act.py"
bias_replacements = [
    ("if not _inited:", "if not _inited:\n        _inited = True"),
    ("warnings.warn('Failed to build CUDA kernels for bias_act. Falling back to slow reference implementation. Details:\n\n' + traceback.format_exc())", "pass # Warning suppressed")
]
patch_file(bias_path, bias_replacements)

print("\n Setup Complete!")

Installing Stylegan2-ada pytorch implementation 
Repository already exists.

 Installing Dependencies
Installing ninja
Reading package lists...
Building dependency tree...
Reading state information...
ninja-build is already the newest version (1.10.1-1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.



 Patching Files for PyTorch 2 Compatibility 
Patched grid_sample_gradfix.py
No changes needed for conv2d_gradfix.py
Patched upfirdn2d.py
Patched bias_act.py

 Setup Complete!


events.out.tfevents.1772728529.48e26663af32.2246.0  network-snapshot-000000.pkl
fakes000000.png					    network-snapshot-000020.pkl
fakes000020.png					    network-snapshot-000040.pkl
fakes000040.png					    network-snapshot-000060.pkl
fakes000060.png					    network-snapshot-000080.pkl
fakes000080.png					    reals.png
fakes_init.png					    stats.jsonl
log.txt						    training_options.json
metric-fid50k_full.jsonl


'/content/drive/MyDrive/stylegan2ada_training_results/00022-bf_hm_combined_dataset-auto1-gamma8-kimg25000-batch4-ada-resumecustom/network-snapshot-000080.pkl'

In [ ]:
import sys
import os
import shutil
import subprocess
# current run -

#FIX: Robust Patch for torch_utils/misc.py
print("--- Checking torch_utils/misc.py ---")
misc_path = "/content/stylegan2-ada-pytorch/torch_utils/misc.py"

if os.path.exists(misc_path):
    with open(misc_path, 'r') as f:
        lines = f.readlines()

    # Check and patch line by line to be sure
    patched = False
    with open(misc_path, 'w') as f:
        for line in lines:
            if "super().__init__(dataset)" in line:
                print(f"Found bad line: {line.strip()}")
                new_line = line.replace("super().__init__(dataset)", "super().__init__()")
                f.write(new_line)
                print(f"Replaced with: {new_line.strip()}")
                patched = True
            else:
                f.write(line)

    if patched:
        print("Successfully patched misc.py")
    else:
        print("No instances of 'super().__init__(dataset)' found. File might already be patched.")

    # Double check
    with open(misc_path, 'r') as f:
        if "super().__init__(dataset)" in f.read():
            print("WARNING: Patch failed! Bad line still exists.")
        else:
            print("Verification: File is clean.")
##############################################################################################################################


##############################################################################################################################
# Force-add Ninja to PATH
ninja_path = subprocess.getoutput(f"{sys.executable} -m pip show -f ninja | grep 'Location' | cut -d ' ' -f 2")
if ninja_path:
    bin_path = os.path.join(ninja_path, 'ninja', 'data', 'bin')
    if os.path.exists(bin_path):
        if bin_path not in os.environ["PATH"]:
            os.environ["PATH"] += os.pathsep + bin_path
            print(f"Added Ninja bin to PATH: {bin_path}")

if "/usr/bin" not in os.environ["PATH"]:
    os.environ["PATH"] += os.pathsep + "/usr/bin"

print(f"Ninja version: {subprocess.getoutput('ninja --version')}")

##############################################################################################################################
# Clear Torch Extensions Cache
torch_extensions_path = os.path.expanduser("~/.cache/torch_extensions")
if os.path.exists(torch_extensions_path):
    print(f"Clearing Torch extensions cache at {torch_extensions_path}...")
    shutil.rmtree(torch_extensions_path)
##############################################################################################################################

In [ ]:
# Train styleGAN2 on T4 runtype in google colab. time intensive

In [ ]:


##############################################################################################################################
#  Training

print("\n Training or Restarting Training")
dataset_zip_path = "/content/drive/MyDrive/bf_hm_combined_dataset.zip"
training_output_dir = "/content/drive/MyDrive/stylegan2ada_training_results"

# setting --batch=4 ato avoid out of memory error, workers 2 often set back to 1
# Comment out --resume with path if you are starting a fresh training of the styleGAN
# kimg is the amount thousand images to be trained through
!{sys.executable} /content/stylegan2-ada-pytorch/train.py \
    --outdir="{training_output_dir}" \
    --data="{dataset_zip_path}" \
    --resume='{resume_path}' \
    --workers=2 \
    --gpus=1 \
    --snap=5 \
    --aug=ada \
    --cfg=auto \
    --batch=4 \
    --gamma=8 \
    --metrics=fid50k_full \
    --kimg=25000

--- Checking torch_utils/misc.py ---
Found bad line: super().__init__(dataset)
Replaced with: super().__init__()
Successfully patched misc.py
Verification: File is clean.
Ninja version: 1.13.0.git.kitware.jobserver-pipe-1

--- Restarting Training (Batch Size Reduced to 4) ---

Training options:
{
  "num_gpus": 1,
  "image_snapshot_ticks": 5,
  "network_snapshot_ticks": 5,
  "metrics": [
    "fid50k_full"
  ],
  "random_seed": 0,
  "training_set_kwargs": {
    "class_name": "training.dataset.ImageFolderDataset",
    "path": "/content/drive/MyDrive/bf_hm_combined_dataset.zip",
    "use_labels": false,
    "max_size": 592,
    "xflip": false,
    "resolution": 512
  },
  "data_loader_kwargs": {
    "pin_memory": true,
    "num_workers": 2,
    "prefetch_factor": 2
  },
  "G_kwargs": {
    "class_name": "training.networks.Generator",
    "z_dim": 512,
    "w_dim": 512,
    "mapping_kwargs": {
      "num_layers": 2
    },
    "synthesis_kwargs": {
      "channel_base": 32768,
      "channel

In [1]:
## Generating synthetic pairs of images

In [16]:
!ls '/content/drive/MyDrive/stylegan2ada_training_results/'

00007-bf_hm_combined_dataset-auto1-kimg25000
00014-bf_hm_combined_dataset-auto1-kimg25000-resumecustom
00016-bf_hm_combined_dataset-auto1-kimg25000-resumecustom
00017-bf_hm_combined_dataset-auto1-kimg25000-batch4-resumecustom
00018-bf_hm_combined_dataset-auto1-kimg25000-batch4-resumecustom
00019-bf_hm_combined_dataset-auto1-gamma6.5536-kimg25000-batch4-ada-resumecustom
00022-bf_hm_combined_dataset-auto1-gamma8-kimg25000-batch4-ada-resumecustom
00023-bf_hm_combined_dataset-auto1-gamma8-kimg25000-batch4-ada-resumecustom


In [17]:
resume_path='/content/drive/MyDrive/stylegan2ada_training_results/00018-bf_hm_combined_dataset-auto1-kimg25000-batch4-resumecustom/network-snapshot-000020.pkl'

In [20]:
%cd /content/stylegan2-ada-pytorch
# Creating 200 synthetic paired iamges
!python generate.py \
  --network={resume_path} \
  --outdir="/content/generated_200" \
  --seeds=0-199

/content/stylegan2-ada-pytorch
Loading networks from "/content/drive/MyDrive/stylegan2ada_training_results/00019-bf_hm_combined_dataset-auto1-gamma6.5536-kimg25000-batch4-ada-resumecustom/network-snapshot-000080.pkl"...
Generating image for seed 0 (0/200) ...
Setting up PyTorch plugin "bias_act_plugin"... Failed!
/content/stylegan2-ada-pytorch/torch_utils/ops/bias_act.py:53: UserWarning: Failed to build CUDA kernels for bias_act. Falling back to slow reference implementation. Details:

Traceback (most recent call last):
  File "/content/stylegan2-ada-pytorch/torch_utils/ops/bias_act.py", line 51, in _init
    _plugin = custom_ops.get_plugin('bias_act_plugin', sources=sources, extra_cuda_cflags=['--use_fast_math'])
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/stylegan2-ada-pytorch/torch_utils/custom_ops.py", line 111, in get_plugin
    module = importlib.import_module(module_name)
             ^^^^^^^^^^^

In [21]:
# move to google drive from colab files
!cp -r /content/generated_200 "/content/drive/MyDrive/GeneratedSytheticPairs18_20"

In [27]:
import os
import numpy as np
from PIL import Image

input_dir = "/content/drive/MyDrive/generated_200_best/Vertical/"
bf_dir = "/content/drive/MyDrive/content/generated_200_brightfield_best"
hm_dir = "/content/drive/MyDrive/content/generated_200_heatmap_best"

os.makedirs(bf_dir, exist_ok=True)
os.makedirs(hm_dir, exist_ok=True)

def crop_nonblack(img, threshold=8):
    arr = np.array(img)
    mask = np.max(arr, axis=2) > threshold
    coords = np.argwhere(mask)

    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0) + 1

    return img.crop((x0, y0, x1, y1))

def split_pair(img):

    w, h = img.size

    # horizontal pair
    if w > h:
        mid = w // 2
        a = img.crop((0, 0, mid, h))
        b = img.crop((mid, 0, w, h))

    # vertical pair
    else:
        mid = h // 2
        a = img.crop((0, 0, w, mid))
        b = img.crop((0, mid, w, h))

    return a, b

for fname in os.listdir(input_dir):

    if not fname.endswith(".png"):
        continue

    img = Image.open(os.path.join(input_dir, fname)).convert("RGB")

    # remove padding
    cropped = crop_nonblack(img)

    # split BF / heatmap
    bf, hm = split_pair(cropped)

    # force final size 256x256
    bf = bf.resize((256,256), Image.BICUBIC)
    hm = hm.resize((256,256), Image.BICUBIC)

    bf.save(os.path.join(bf_dir, fname))
    hm.save(os.path.join(hm_dir, fname))

print("Finished splitting and cropping.")

Finished splitting and cropping.


In [ ]:
# supplmenetatl script, creating a zip dataset for the styleGAN based on a paired BF/Sarcomere iamge and their heatmap

In [ ]:


from google.colab import drive
import os
from PIL import Image

drive.mount("/content/drive")

bf_dir = "/content/drive/MyDrive/BF_HM_FL/AlignedOutputs/overlay/"       # BF brightfield
hm_dir = "/content/drive/MyDrive/BF_HM_FL/NoContourHM_converted/"        # HM heatmap

out_root = "/content/drive/MyDrive/stylegan2ada_bf_hm_pairs_256x512"
pairs_dir = f"{out_root}/pairs_images"     # paired combo images live here
os.makedirs(pairs_dir, exist_ok=True)

print("BF:", bf_dir)
print("HM:", hm_dir)
print("OUT:", out_root)

print("checking if we can create apaired image dataset")


bf_image_paths = sorted([os.path.join(bf_dir, f) for f in os.listdir(bf_dir) if os.path.isfile(os.path.join(bf_dir, f))])
print(f"Found {len(bf_image_paths)} brightfield image files.")

hm_image_paths = sorted([os.path.join(hm_dir, f) for f in os.listdir(hm_dir) if os.path.isfile(os.path.join(hm_dir, f))])
print(f"Found {len(hm_image_paths)} heatmap image files.")

# Check if counts match for pairing
if len(bf_image_paths) != len(hm_image_paths):
    print("Error: Number of BF and HM images do not match. Cannot perform sequential pairing.")
    paired_images_to_process = []
else:
    # Pair them sequentially
    paired_images_to_process = list(zip(bf_image_paths, hm_image_paths))
    print(f"Found {len(paired_images_to_process)} pairs to process.")

#  pairs_dir exists
os.makedirs(pairs_dir, exist_ok=True)
print(f"Ensured paired images directory exists: {pairs_dir}")

combined_count = 0
for i, (bf_path, hm_path) in enumerate(paired_images_to_process):
    # Extract base name for output file using BF's base name as primary id
    base_name = os.path.splitext(os.path.basename(bf_path))[0]
    output_image_path = os.path.join(pairs_dir, f'{base_name}_combined.png')

    try:
        # Open and convert to RGB
        bf_image = Image.open(bf_path).convert('RGB')
        hm_image = Image.open(hm_path).convert('RGB')

        # Create new image with double width
        combined_image = Image.new('RGB', (bf_image.width + hm_image.width, bf_image.height))

        # Paste images
        combined_image.paste(bf_image, (0, 0))
        combined_image.paste(hm_image, (bf_image.width, 0))

        # Save combined image
        combined_image.save(output_image_path)
        # print(f"Saved combined image: {output_image_path}") # Uncomment for verbose output
        combined_count += 1
    except FileNotFoundError:
        print(f"Warning: Skipping pair {base_name} due to missing file in its original location.")
    except Exception as e:
        print(f"Error processing pair {base_name}: {e}")

print(f"Finished creating paired images. Total combined images created: {combined_count}")
print("Paired image dataset done, now to zip")
import sys
import os

# Define the full path for the output .zip dataset
dataset_zip_path = "/content/drive/MyDrive/bf_hm_combined_dataset.zip" # put wherever, used for trianing the styleGAN

print(f"creating paired images to StyleGAN2-ADA dataset zip")
print(f"Source directory for images: {pairs_dir}")
print(f"Destination .zip dataset: {dataset_zip_path}")

# Execute the dataset_tool.py script using its absolute path
# The repo /content/stylegan2-ada-pytorch
# Explicitly set resolution as the images are now square and consistent
!{sys.executable} /content/stylegan2-ada-pytorch/dataset_tool.py --source="{pairs_dir}" --dest="{dataset_zip_path}" --resolution=512

print(f"StyleGAN2-ADA dataset created successfully at {dataset_zip_path}")